# geodetic-engine -- Catalog examples

`geodetic_engine.catalog` lists what the active PROJ database holds -- CRSs, coordinate operations, datums and datum ensembles, ellipsoids, prime meridians, units, operation methods, coordinate systems and grid files -- and filters them with one consistent set of keyword arguments.

Each listing function returns a lazy `Listing`: nothing is read until it is used, and the entries can be looped over, turned into a pandas DataFrame, or exported as PROJJSON-shaped JSON. Every entry's `to_json_dict()` is the object's official PROJJSON, as PROJ exports it.

The catalog lists what the registry *publishes*. To see what PROJ would *run* between two CRSs, use `geodetic_engine.geodesy.available_operations`; Section 4 compares the two.

## 1. Which database answers

Every listing reads the `proj.db` PROJ resolves codes against. `database()` names it, with the dataset versions it holds, and `authorities()` lists the authorities any `authority=` filter may name.

In [2]:
from geodetic_engine import catalog

info = catalog.database()
print("PROJ", info.proj_version, "| EPSG", info.epsg_version)
print([str(authority) for authority in catalog.authorities()])

PROJ 9.9.0 | EPSG v13.102
['EPSG v13.102', 'ESRI ArcGIS Pro 3.6', 'Equinor', 'IAU_2015', 'IGNF 3.1.0', 'NKG 1.0.w', 'NRCAN', 'OGC', 'OSDU', 'PROJ 9.9.0']


## 2. Projected CRSs on a datum, for an area

The question the catalog answers most often. `area_of_interest` is `(west, south, east, north)` in degrees, and PROJ itself decides which areas of use intersect it. This is the example from the issue that asked for the catalog:

In [3]:
european = catalog.crs(
    authority="EPSG",
    kind="projected",
    datum="ETRS89",
    area_of_interest=(-10, 50, 30, 72),
)
print(len(european), "projected ETRS89 CRSs intersect the area")
european.to_dataframe()[["authority_code", "name", "projection_method"]].head(10)

41 projected ETRS89 CRSs intersect the area


,authority_code,name,projection_method
0,EPSG:2157,ETRS89-IRE [ETRF2000] / Irish Transverse Mercator,Transverse Mercator
1,EPSG:2158,ETRS89-IRE [ETRF2000] / UTM zone 29N,Transverse Mercator
2,EPSG:2180,ETRS89 / PL-1992,Transverse Mercator
3,EPSG:2196,ETRS89 / Kp2000 Jutland,Transverse Mercator
4,EPSG:2197,ETRS89 / Kp2000 Zealand,Transverse Mercator
5,EPSG:2198,ETRS89 / Kp2000 Bornholm,Transverse Mercator
6,EPSG:3034,ETRS89-extended / LCC Europe,Lambert Conic Conformal (2SP)
7,EPSG:3035,ETRS89-extended / LAEA Europe,Lambert Azimuthal Equal Area
8,EPSG:3041,ETRS89 / UTM zone 29N (N-E),Transverse Mercator
9,EPSG:3042,ETRS89 / UTM zone 30N (N-E),Transverse Mercator


Narrowed to UTM zones whose area of use *contains* a box around Bergen, with a pattern on the name:

In [4]:
import re

bergen = (5.0, 60.2, 5.6, 60.6)
for entry in catalog.crs(
    authority="EPSG",
    kind="projected",
    datum="ETRS89",
    name=re.compile(r"UTM zone \d+N$"),
    area_of_interest=bergen,
    contains=True,
):
    print(entry, "|", entry.area_of_use.name[:60])

ETRS89 / UTM zone 31N (EPSG:25831) | Europe between 0°E and 6°E: Andorra; Denmark - offshore; Ger


## 3. Transformations between two CRSs

`operations()` with `source_crs` and `target_crs` lists the operations registered between them, each with its method, stated accuracy, version label and area of use. The second example from the issue:

In [22]:
ed50 = catalog.operations(
    source_crs="EPSG:4230", target_crs="EPSG:4326", max_accuracy=5.0
)
print(len(ed50), "ED50 -> WGS 84 transformations state 5 m or better")
ed50.to_dataframe()[
    ["authority_code", "method", "accuracy", "version", "area_of_use"]
].head(12)

25 ED50 -> WGS 84 transformations state 5 m or better


,authority_code,method,accuracy,version,area_of_use
0,EPSG:1087,Geocentric translations (geog2D domain) (EPSG:...,2.5,RJGC-Jor,Jordan.
1,EPSG:1275,Geocentric translations (geog2D domain) (EPSG:...,2.0,IGN-Fra,"France - onshore and offshore, mainland and Co..."
2,EPSG:1311,Position Vector transformation (geog2D domain)...,1.0,UKOOA-CO,Denmark - offshore North Sea; Ireland - offsho...
3,EPSG:1612,Position Vector transformation (geog2D domain)...,1.0,EPSG-Nor N62 2001,Norway - offshore north of 62°N. Also Svalbard...
4,EPSG:1613,Position Vector transformation (geog2D domain)...,1.0,EPSG-Nor S62 2001,Norway - offshore south of 62°N - North Sea.
5,EPSG:1627,Position Vector transformation (geog2D domain)...,1.0,EPSG-Dnk,Denmark - onshore.
6,EPSG:1629,Geocentric translations (geog2D domain) (EPSG:...,1.0,EPSG-Gib,Gibraltar - onshore and offshore.
7,EPSG:1631,Position Vector transformation (geog2D domain)...,1.5,EPSG-Esp Bal,Spain - Balearic Islands.
8,EPSG:1633,Position Vector transformation (geog2D domain)...,1.5,EPSG-Esp,Spain - onshore mainland except northwest (nor...
9,EPSG:1635,Position Vector transformation (geog2D domain)...,1.5,EPSG-Esp NW,Spain - onshore mainland north of 41°30'N and ...


In [6]:
by_grid = catalog.operations(
    source_crs="EPSG:4230", target_crs="EPSG:4326", grid_based=True
)
for entry in by_grid:
    print(
        f"{entry.authority_code:10} {entry.method.name:12} grids={entry.grid_names}  available={entry.grids_available()}"
    )

EPSG:15933 NTv2         grids=('SPED2ETV2.gsb',)  available=False
ESRI:108068 NTv2         grids=('PENR2009',)  available=False
ESRI:108069 NTv2         grids=('BALR2009',)  available=False
ESRI:108119 NTv2         grids=('baleares',)  available=False
ESRI:108136 NTv2         grids=('peninsula',)  available=False
ESRI:108359 NTv2         grids=('100800401',)  available=False


Choosing one for a point offshore Norway: the most accurate transformation whose area of use contains it.

In [7]:
point = (4.1279, 63.5850)
candidates = catalog.operations(
    source_crs="EPSG:4230",
    target_crs="EPSG:4326",
    area_of_interest=(*point, *point),
    contains=True,
    max_accuracy=10,
)
best = min(candidates, key=lambda entry: entry.accuracy)
print("chosen:", best, best.accuracy, "m")
print("valid in:", best.area_of_use.name)

chosen: ED50 to WGS 84 (18) (EPSG:1311) 1.0 m
valid in: Denmark - offshore North Sea; Ireland - offshore; Netherlands - offshore; United Kingdom - UKCS offshore.


That is a UK offshore transformation: its bounding box reaches the point, but its area of use, in words, does not. A bounding box is the rectangle around an area, so read the area's name before trusting it. Adding `area_name` settles it, and the entry's code is an `operation=` for `transform()`:

In [8]:
from geodetic_engine.geodesy import transform

best = min(
    catalog.operations(
        source_crs="EPSG:4230",
        target_crs="EPSG:4326",
        area_of_interest=(*point, *point),
        contains=True,
        area_name="Norway",
    ),
    key=lambda entry: entry.accuracy,
)
print("chosen:", best, best.accuracy, "m |", best.area_of_use.name)
transform("EPSG:4230", "EPSG:4326", point, operation=best.authority_code).coordinates

chosen: ED50 to WGS 84 (23) (EPSG:1612) 1.0 m | Norway - offshore north of 62°N. Also Svalbard - onshore and offshore.


((4.126139263192541, 63.5846361020684),)

## 4. Registered, or assembled by PROJ

EPSG registers datum transformations between geographic CRSs. Between two projected CRSs nothing is registered, but PROJ assembles a chain: the inverse projection, a registered transformation, the projection. The catalog answers the first question, `available_operations` the second:

In [9]:
from geodetic_engine.geodesy import available_operations

print(
    "registered:",
    len(catalog.operations(source_crs="EPSG:23031", target_crs="EPSG:32631")),
)
for candidate in available_operations("EPSG:23031", "EPSG:32631")[:3]:
    print("assembled: ", candidate.references, candidate.accuracy, "m")

registered: 0
assembled:  ('EPSG:16031', 'EPSG:1133', 'EPSG:16031') 10.0 m
assembled:  ('EPSG:16031', 'ESRI:108335', 'EPSG:16031') 10.0 m
assembled:  ('EPSG:16031', 'EPSG:1612', 'EPSG:16031') 1.0 m


PROJ never offers a deprecated operation. The catalog lists them on request, with what replaces them:

In [10]:
for entry in catalog.operations(
    source_crs="EPSG:4230", target_crs="EPSG:4326", deprecated=True
):
    print(entry, "-> replaced by", entry.replaced_by or "nothing stated")

ED50 to WGS 84 (32) (EPSG:1961) -> replaced by nothing stated
ED50 to WGS 84 (32) (EPSG:1999) -> replaced by nothing stated
ED50 to WGS 84 (40) (EPSG:15907) -> replaced by nothing stated
ST_ED50_WGS84_N65_T8047 (Equinor:3000037) -> replaced by nothing stated


## 5. Datums, ensembles and frames

`datums()` lists geodetic, vertical and engineering datums and datum ensembles. An ensemble lists its members, and `member_of` turns that round:

In [23]:
etrs89 = catalog.datums(code="EPSG:6258").one()
print(
    etrs89.projjson_type,
    "| accuracy",
    etrs89.ensemble_accuracy,
    "m |",
    len(etrs89.members),
    "members",
)
[str(member) for member in catalog.datums(member_of="ETRS89")][:8]

DatumEnsemble | accuracy 0.1 m | 60 members


['MOLDREF99 (EPSG:1032)',
 'Serbian Reference Network 1998 (EPSG:1034)',
 'Rete Dinamica Nazionale 2008 (EPSG:1132)',
 'Bulgaria Geodetic System 2005 (EPSG:1167)',
 'European Terrestrial Reference Frame 1989 (EPSG:1178)',
 'European Terrestrial Reference Frame 1990 (EPSG:1179)',
 'European Terrestrial Reference Frame 1991 (EPSG:1180)',
 'European Terrestrial Reference Frame 1992 (EPSG:1181)']

In [12]:
dynamic = catalog.datums(
    authority="EPSG", dynamic=True, frame_reference_epoch=(2014, None)
)
[(str(d), d.frame_reference_epoch) for d in dynamic]

[('SIRGAS Continuously Operating Network SIR17P01 (EPSG:1242)', 2015.0),
 ('Australian Terrestrial Reference Frame 2014 (EPSG:1291)', 2020.0),
 ('Red Geodesica Para Mineria en Chile (EPSG:1304)', 2024.0),
 ('World Geodetic System 1984 (G2139) (EPSG:1309)', 2016.0),
 ('International Terrestrial Reference Frame 2020 (EPSG:1322)', 2015.0),
 ('IGS20 (EPSG:1333)', 2015.0),
 ('World Geodetic System 1984 (G2296) (EPSG:1383)', 2024.0)]

## 6. Following references

Every entry points at the objects it is built from with an `ObjectRef`, and every `code` filter accepts one, so following a reference is one more listing. From a CRS to its datum, to its ellipsoid, to every other datum on that ellipsoid:

In [13]:
crs = catalog.crs(code="EPSG:23031").one()
datum = catalog.datums(code=crs.datums).one()
ellipsoid = catalog.ellipsoids(code=datum.ellipsoid).one()
others = catalog.datums(ellipsoid=ellipsoid)
print(crs, "->", datum, "->", ellipsoid)
print(
    f"a={ellipsoid.semi_major_axis} m, 1/f={ellipsoid.inverse_flattening}; {len(others)} datums use it"
)

ED50 / UTM zone 31N (EPSG:23031) -> European Datum 1950 (EPSG:6230) -> International 1924 (EPSG:7022)
a=6378388.0 m, 1/f=297.0; 216 datums use it


## 7. Ellipsoids, prime meridians and units

In [14]:
catalog.ellipsoids(authority="EPSG", semi_major_axis=(6378100, 6378200)).to_dataframe()[
    ["authority_code", "name", "semi_major_axis", "inverse_flattening"]
]

,authority_code,name,semi_major_axis,inverse_flattening
0,EPSG:1024,CGCS2000,6378137.0,298.257222
1,EPSG:1025,GSK-2011,6378136.5,298.256415
2,EPSG:7003,Australian National Spheroid,6378160.0,298.250000
3,EPSG:7019,GRS 1980,6378137.0,298.257222
4,EPSG:7020,Helmert 1906,6378200.0,298.300000
5,EPSG:7021,Indonesian National Spheroid,6378160.0,298.247000
6,EPSG:7025,NWL 9D,6378145.0,298.250000
7,EPSG:7030,WGS 84,6378137.0,298.257224
8,EPSG:7031,GEM 10C,6378137.0,298.257224
9,EPSG:7032,OSU86F,6378136.2,298.257224


In [15]:
catalog.prime_meridians(authority="EPSG", longitude=(-20, 20)).to_dataframe()[
    ["name", "longitude", "unit", "longitude_degrees"]
]

,name,longitude,unit,longitude_degrees
0,Copenhagen,12.577750,degree,12.577750
1,Greenwich,0.000000,degree,0.000000
2,Lisbon,-9.131906,degree,-9.131906
3,Paris,2.596921,grad,2.337229
4,Madrid,-3.687375,degree,-3.687375
5,Rome,12.452333,degree,12.452333
6,Bern,7.439583,degree,7.439583
7,Ferro,-17.666667,degree,-17.666667
8,Brussels,4.367975,degree,4.367975
9,Stockholm,18.058278,degree,18.058278


In [16]:
catalog.units(authority="EPSG", kind="linear", name="foot").to_dataframe()[
    ["authority_code", "name", "conversion_factor"]
]

,authority_code,name,conversion_factor
0,EPSG:9002,foot,0.304800
1,EPSG:9003,US survey foot,0.304801
2,EPSG:9005,Clarke's foot,0.304797
3,EPSG:9041,British foot (Sears 1922),0.304799
4,EPSG:9051,British foot (Benoit 1895 A),0.304800
5,EPSG:9061,British foot (Benoit 1895 B),0.304800
6,EPSG:9070,British foot (1865),0.304801
7,EPSG:9080,Indian foot,0.304800
8,EPSG:9081,Indian foot (1937),0.304798
9,EPSG:9082,Indian foot (1962),0.304800


## 8. Methods and coordinate systems

A method lists the parameters PROJ states for it, and how many operations apply it. A coordinate system lists its axes; `axes` finds the northing-first ones, whose CRSs are easiest to get wrong:

In [17]:
tm = catalog.methods(code="EPSG:9807").one()
print(tm, "|", tm.operation_count, "operations")
print([p.name for p in tm.parameters])
[
    str(cs)
    for cs in catalog.coordinate_systems(
        authority="EPSG", axes=("N", "E"), unit="metre"
    )
]

Transverse Mercator (EPSG:9807) | 2602 operations
['Latitude of natural origin', 'Longitude of natural origin', 'Scale factor at natural origin', 'False easting', 'False northing']


['Cartesian 2D CS. Axes: northing, easting (N,E). Orientations: North along 180°E,North along 90°W. UoM: metre. (EPSG:1044)',
 'Cartesian 2D CS. Axes: local northing, local easting (n,e). Orientations: north,east. UoM: metre. (EPSG:1047)',
 'Cartesian 2D CS. Axes: northing, easting (N,E). Orientations: South along 180°E,South along 90°E. UoM: metre. (EPSG:4493)',
 'Cartesian 2D CS. Axes: northing, easting (N,E). Orientations: North along 0°E,North along 90°E. UoM: metre. (EPSG:4494)',
 'Cartesian 2D CS. Axes: northing, easting (N,E). Orientations: north,east. UoM: metre. (EPSG:4500)',
 'Cartesian 2D CS. Axes: northing, westing (N,E). Orientations: north,west. UoM: metre. (EPSG:4501)',
 'Cartesian 2D CS. Axes: plant north, plant east (n,e). Orientations: northWest,northEast. UoM: metre. (EPSG:6504)',
 'Cartesian 2D CS. Axes: first local axis, second local axis (n,e). Orientations: northWest,northEast. UoM: metre. (EPSG:6505)']

## 9. Grid files

`grids()` lists the grid files operations read, under the file PROJ reads for each, and whether it is installed. Here, the grids for Norway, and the operations that need a missing one:

In [18]:
norway = catalog.grids(area_name="Norway")
norway.to_dataframe()[["name", "kind", "installed", "used_by"]]

,name,kind,installed,used_by
0,BSCD2000.txt,vertical,False,"(EPSG:10680, EPSG:10681)"
1,ChartDatum_above_Ellipsoid_EUREF89_v2021b.bin,vertical,False,"(EPSG:10130, EPSG:10133, EPSG:11568, EPSG:11569)"
2,ChartDatum_above_Ellipsoid_EUREF89_v2023a.bin,vertical,False,"(EPSG:10504, EPSG:10505, EPSG:11570, EPSG:11571)"
3,eur_nkg_nkgrf03vel_realigned.tif,velocity,True,"(NKG:ETRF00_TO_DK, NKG:ETRF00_TO_EE, NKG:ETRF0..."
4,eur_nkg_nkgrf17vel.tif,velocity,True,"(EPSG:10808, EPSG:10809, EPSG:10811, EPSG:1081..."
5,no_kv_CD_above_Ell_ETRS89_v2021a.tif,vertical,True,"(EPSG:9884, EPSG:9885, EPSG:11566, EPSG:11567)"
6,no_kv_CD_above_Ell_ETRS89_v2023b.tif,vertical,True,"(EPSG:10509, EPSG:10510, EPSG:11572, EPSG:11573)"
7,no_kv_ETRS89NO_NGO48_TIN.json,horizontal,True,"(EPSG:10854, EPSG:11555, Equinor:3000044, Equi..."
8,no_kv_href2008a.tif,vertical,True,"(EPSG:9484, EPSG:9594, EPSG:11559, EPSG:11561)"
9,no_kv_HREF2018B_NN2000_EUREF89.tif,vertical,True,"(EPSG:9485, EPSG:9593, EPSG:11398, EPSG:11556,..."


In [19]:
missing = [grid for grid in norway if not grid.installed]
codes = [code for grid in missing for code in grid.used_by]
if codes:
    print([str(entry) for entry in catalog.operations(code=codes, deprecated=None)])
else:
    print("every grid these operations read is installed")

['ETRS89 to CD Norway depth (2) (EPSG:10130)', 'ETRS89 to ETRS89 + CD Norway depth (2) (EPSG:10133)', 'ETRS89 to CD Norway depth (3) (EPSG:10504)', 'ETRS89 to ETRS89 + CD Norway depth (3) (EPSG:10505)', 'ETRS89 to BSCD2000 depth (1) (EPSG:10680)', 'ETRS89 to ETRS89 + BSCD2000 depth (1) (EPSG:10681)', 'ETRS89-NOR [EUREF89] to CD Norway depth (2) (EPSG:11568)', 'ETRS89-NOR [EUREF89] to ETRS89-NOR [EUREF89] + CD Norway depth (2) (EPSG:11569)', 'ETRS89-NOR [EUREF89] to CD Norway depth (3) (EPSG:11570)', 'ETRS89-NOR [EUREF89] to ETRS89-NOR [EUREF89] + CD Norway depth (3) (EPSG:11571)']


`check_remote=True` also asks PROJ's content delivery network which grids it publishes. It is the only listing that uses the network, so it is not run here.

## 10. Exporting

A listing exports as JSON: PROJJSON-shaped items in a small envelope recording the database that answered and the filters asked for. `include_definitions=True` adds each entry's full PROJJSON. The envelope's JSON Schema ships with the package as `catalog.listing_schema()`.

In [20]:
import json

document = catalog.operations(code="EPSG:1612").to_json_dict()
print(json.dumps(document["items"][0], indent=2))
print(document["filters"], document["database"]["metadata"]["EPSG.VERSION"])

{
  "type": "Transformation",
  "name": "ED50 to WGS 84 (23)",
  "id": {
    "authority": "EPSG",
    "code": 1612
  },
  "kind": "transformation",
  "area": "Norway - offshore north of 62\u00b0N. Also Svalbard - onshore and offshore.",
  "bbox": {
    "south_latitude": 62.0,
    "west_longitude": -3.35,
    "north_latitude": 84.73,
    "east_longitude": 38.01
  },
  "method": {
    "name": "Position Vector transformation (geog2D domain)",
    "id": {
      "authority": "EPSG",
      "code": 9606
    }
  },
  "source_crs": {
    "name": "ED50",
    "id": {
      "authority": "EPSG",
      "code": 4230
    }
  },
  "target_crs": {
    "name": "WGS 84",
    "id": {
      "authority": "EPSG",
      "code": 4326
    }
  },
  "accuracy": "1.0",
  "version": "EPSG-Nor N62 2001",
  "osdu_ids": [
    "reference-data--CoordinateTransformation:EPSG::1612"
  ],
  "has_persistable_reference": true,
  "deprecated": false,
  "aliases": [
    "ED_1950_To_WGS_1984_23",
    "reference-data--CoordinateT

In [21]:
definition = catalog.operations(code="EPSG:1612").one().to_json_dict()
print(definition["$schema"])
[(parameter["name"], parameter["value"]) for parameter in definition["parameters"]]

https://proj.org/schemas/v0.7/projjson.schema.json


[('X-axis translation', -116.641),
 ('Y-axis translation', -56.931),
 ('Z-axis translation', -110.559),
 ('X-axis rotation', 0.893),
 ('Y-axis rotation', 0.921),
 ('Z-axis rotation', -0.917),
 ('Scale difference', -3.52)]